In [ ]:
# Define the model class
class LightModel(pl.LightningModule):
    # Define a linear layer to transform your input
    def __init__(self):
        super().__init__()
        self.layer = torch.nn.Linear(16, 10)
    def forward(self, x):
        return self.layer(x)
    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = torch.nn.functional.cross_entropy(logits, y)
        return loss

In [ ]:
from lightning.pytorch import Trainer
# Configure the Trainer with max epochs
trainer = Trainer(max_epochs=3, devices=1)
# Start training with the predefined model
trainer.fit(model, train_dataloader, val_dataloader)

In [ ]:
import lightning.pytorch as pl
import torch.nn as nn

# Create the class
class ClassifierModel(pl.LightningModule):
    # Create init method
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.classifier = nn.Linear(input_dim, output_dim)

In [ ]:
class ClassifierModel(pl.LightningModule):
  
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.hidden = nn.Linear(input_dim, hidden_dim)
        self.output = nn.Linear(hidden_dim, output_dim)
        
    # Define forward method
    def forward(self, x):
        # Complete the forward pass
        x = self.hidden(x)
        x = nn.ReLU(x)
        x = self.output(x)
        return x

In [ ]:
from torch.nn.functional import cross_entropy

def training_step(self, batch, batch_idx):
    x, y = batch
    # Ensure that you compute predictions using the forward pass
    y_hat = self(x)
    # Calculate the cross entropy loss
    loss = cross_entropy(y_hat, y)
    # Log the loss
    self.log("train_loss", loss)
    return loss

In [ ]:
import torch

def configure_optimizers(self):
  	# Create an Adam optimizer for model parameters
    optimizer = torch.optim.Adam(self.parameters(), lr=1e-3)
    return optimizer

In [ ]:
# Import the Trainer
from lightning.pytorch import Trainer

# Define ImageClassifier model & trainer and set epoch parameter
model = ImageClassifier()
trainer = Trainer(max_epochs=5)

# Train the model
trainer.fit(model, train_loader, val_loader)

# Evaluate the model
val_results = trainer.validate(model, val_loader)
print("Validation Accuracy:", val_results[0]["val_acc"])

In [ ]:
# Import libraries
import lightning.pytorch as pl
from torch.utils.data import random_split

class SplitDataModule(pl.LightningDataModule):
    def __init__(self):
        super().__init__()
        self.train_data = None
        self.val_data = None
    def setup(self, stage=None):
        # Split the dataset into training (80%) and validation (20%)
        self.train_data, self.val_data = random_split(dataset, [80, 20])

In [ ]:
# Import libraries
from torch.utils.data import DataLoader
import lightning.pytorch as pl

class LoaderDataModule(pl.LightningDataModule):
    def __init__(self):
        super().__init__()
        self.train_data = None
        self.val_data = None
    def setup(self, stage=None):
        self.train_data, self.val_data = random_split(dataset, [80, 20])
    def train_dataloader(self):
      	# Complete DataLoader
        return DataLoader(self.train_data, batch_size=16, shuffle=True) 

In [ ]:
import torch.nn.functional as F

def validation_step(self, batch, batch_idx):
    x, y = batch
    # Compute predictions using the model
    preds = self(x)
    # Calculate validation loss
    loss = F.cross_entropy(preds, y)
    # Log the validation loss
    self.log('val_loss', loss)

In [ ]:
# Import relevant metric
from torchmetrics import Accuracy
import lightning.pytorch as pl

class ClassifierModel(pl.LightningModule):
    def __init__(self):
        super().__init__()
        # Instantiate accuracy metric
        self.accuracy = Accuracy()
    def validation_step(self, batch, batch_idx):
        x, y = batch
        preds = self(x)
        # Calculate accuracy and log it as val_acc
        acc = self.accuracy(preds, y)
        self.log('val_acc', acc)

In [ ]:
# Import relevant checkpoints
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping

class EvaluatedImageClassifier(ImageClassifier):
    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        acc = (y_hat.argmax(dim=1) == y).float().mean()
        self.log("val_acc", acc)

model = EvaluatedImageClassifier()
data_module = OsmanyaDataModule()
# Train the model with ModelCheckpoint and EarlyStopping checkpoints
trainer = Trainer(callbacks=[ModelCheckpoint(monitor="val_acc", save_top_k=1), EarlyStopping(monitor="val_acc", patience=3)])
trainer.fit(model, datamodule=data_module)

In [ ]:
import torch
# Import the necessary quantization module
from torch.quantization import quantize_dynamic

# Apply dynamic quantization targeting linear layers
model_quantized = torch.quantization.quantize_dynamic(
    model, {torch.nn.Linear}, dtype=torch.qint8
)

In [ ]:
# Measure inference time of the original model
original_time = measure_time(model, test_loader)

# Measure inference time of the quantized model
quant_time = measure_time(model_quantized, test_loader)

# Print results
print(f"Original Model Time: {original_time:.2f}s")
print(f"Quantized Model Time: {quant_time:.2f}s")

In [ ]:
# Import pruning module
import torch.nn.utils.prune as prune
# Before pruning
print(model)
# Apply L1 unstructured pruning to model[3]
prune.l1_unstructured(model[3], name="weight", amount=0.3)
# After pruning
print(model)

In [ ]:
import torch.nn.utils.prune as prune
# Before pruning
print(model)
# Finalize pruning by removing the pruning mask
prune.remove(model[3], "weight")
# Print model structure after pruning
print(model)

In [ ]:
# Export model to TorchScript
scripted_model = torch.jit.trace(model, torch.tensor(X_test[:1], dtype=torch.float32).unsqueeze(1))
# Save model to TorchScript
torch.jit.save(scripted_model, 'model.pt')

# Loaded saved model
loaded_model = torch.jit.load('model.pt')
# Validate inference on test dataset
test_loader = DataLoader(TensorDataset(torch.tensor(X_test, dtype=torch.float32).unsqueeze(1), y_test), batch_size=64)

accuracy = evaluate_model(loaded_model, test_loader)

print(f"Optimized model accuracy: {accuracy:.2%}")